# Week 3 lab — is the simulated staffing effect larger than Monte Carlo error?

**Driving question.** Under the fixed Week 2 scenario, what is the expected reduction in mean waiting time when staffing increases from 16 to 17 agents, and how can we estimate it using simulations?

One simulation gives one possible 09:00–11:00 morning. This week we repeat the complete morning, distinguish variation between mornings from Monte Carlo error, and attach an approximate confidence interval to the expected staffing effect.

The result remains conditional on the Week 2 assumptions: the selected VRU entries are treated as requests for human service, service times are independent exponential variables with mean 6 minutes, agents are identical, service is FIFO, callers do not abandon, and each morning starts empty at 09:00. 

**Work in your ignored copy.** From the repository root:

```bash
mkdir -p students
cp notebooks/lecture3_simulation_error_control.ipynb students/lecture3_simulation_error_control_work.ipynb
```

Open the copy using the local Python environment from Week 1. Start Jupyter or VS Code from the repository root so the loader can find `data/`. The notebook uses only Python's standard library. See the [AnonymousBank data guide](../data/anonymousbank_data_guide.md) for the source and field definitions.


## Learning goals

By the end of the lab, you should be able to:

- distinguish caller outcomes inside one morning from independent simulation repetitions;
- compare 16 and 17 agents with identical arrivals and service requirements;
- distinguish the standard deviation of morning outcomes from the standard error of an estimated mean;
- connect the law of large numbers and central limit theorem to a simulation experiment;
- plan a number of repetitions and report model sensitivity separately from Monte Carlo error.


In [1]:
import csv
import math
import random
import statistics
from datetime import datetime
from pathlib import Path

## 1. Recreate the fixed Week 2 scenario

The next cells collect the established Week 2 code in one place. 

The arrival model was fitted on ten regular working days. For each simulated morning it draws a shared day multiplier and then generates arrivals from a constant-rate Poisson process over 09:00–11:00. Later we will repeat the analysis with the whole-morning bootstrap generator.

In [2]:
filename = "anonymousbank_nov1999_arrivals_15min.csv"
candidates = [
    Path.cwd() / "data" / filename,
    Path.cwd().parent / "data" / filename,
]
data_path = next((path for path in candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Start from the repository root or from notebooks/.")

def minutes_from_midnight(clock):
    hour, minute = map(int, clock.split(":"))
    return 60 * hour + minute

with data_path.open(newline="", encoding="utf-8") as handle:
    rows = [
        {
            "date": row["date"],
            "minute": minutes_from_midnight(row["time"]),
            "entries": int(row["entries"]),
        }
        for row in csv.DictReader(handle)
    ]

assert len(rows) == 30 * 96
assert sum(row["entries"] for row in rows) == 40_722

# In Israel, the regular working days represented here are Sunday–Thursday.
regular_dates = sorted(
    {
        row["date"]
        for row in rows
        if datetime.strptime(row["date"], "%Y-%m-%d").weekday()
        not in (4, 5)
    }
)
training_dates = regular_dates[:10]

morning_counts = [
    row["entries"]
    for row in rows
    if row["date"] in training_dates
    and 9 * 60 <= row["minute"] < 11 * 60
]
fitted_morning_count = statistics.mean(morning_counts)

daily_totals = [
    sum(row["entries"] for row in rows if row["date"] == day)
    for day in training_dates
]
daily_mean = statistics.mean(daily_totals)
daily_variance = statistics.variance(daily_totals)
alpha = (daily_variance - daily_mean) / daily_mean**2

morning_profiles = {
    day: sorted(
        [
            row
            for row in rows
            if row["date"] == day
            and 9 * 60 <= row["minute"] < 11 * 60
        ],
        key=lambda row: row["minute"],
    )
    for day in training_dates
}

print(f"Loaded {len(rows):,} quarter-hour counts from {data_path}")
print(f"Training dates: {len(training_dates)}")
print(f"Fitted morning count per 15 minutes: {fitted_morning_count:.2f}")
print(f"Estimated Var(G_d) = alpha: {alpha:.5f}")


Loaded 2,880 quarter-hour counts from c:\Users\PR069694\Documents\stochastic-modeling-simulation-ds\data\anonymousbank_nov1999_arrivals_15min.csv
Training dates: 10
Fitted morning count per 15 minutes: 39.85
Estimated Var(G_d) = alpha: 0.01825


In [3]:
def simulate_queue(arrivals, services, number_of_agents):
    if not isinstance(number_of_agents, int) or number_of_agents <= 0:
        raise ValueError("number_of_agents must be a positive integer")
    if len(arrivals) != len(services):
        raise ValueError("arrivals and services must have equal length")
    if any(arrival < 0 for arrival in arrivals):
        raise ValueError("arrival times must be non-negative")
    if any(service <= 0 for service in services):
        raise ValueError("service times must be positive")
    if any(later < earlier for earlier, later in zip(arrivals, arrivals[1:])):
        raise ValueError("arrivals must be sorted")

    agent_free = [0.0] * number_of_agents
    records = []
    for customer, (arrival, service) in enumerate(
        zip(arrivals, services), start=1
    ):
        agent_index = min(
            range(number_of_agents),
            key=lambda index: (agent_free[index], index),
        )
        start = max(arrival, agent_free[agent_index])
        wait = start - arrival
        departure = start + service
        agent_free[agent_index] = departure
        records.append(
            {
                "customer": customer,
                "arrival": arrival,
                "wait": wait,
                "start": start,
                "service": service,
                "departure": departure,
                "agent": agent_index + 1,
            }
        )
    return records

def generate_parametric_morning(rng, horizon=120.0):
    multiplier = rng.gammavariate(1 / alpha, alpha)
    rate_per_minute = multiplier * fitted_morning_count / 15
    arrivals = []
    clock = 0.0
    while True:
        clock += rng.expovariate(rate_per_minute)
        if clock >= horizon:
            break
        arrivals.append(clock)
    return arrivals, multiplier

def generate_bootstrap_morning(rng):
    selected_date = rng.choice(sorted(morning_profiles))
    arrivals = []
    for row in morning_profiles[selected_date]:
        interval_start = row["minute"] - 9 * 60
        arrivals.extend(
            interval_start + 15 * rng.random()
            for _ in range(row["entries"])
        )
    return sorted(arrivals), selected_date

def generate_services(number_of_calls, rng, mean_service=6.0):
    rate_per_minute = 1 / mean_service
    return [
        rng.expovariate(rate_per_minute)
        for _ in range(number_of_calls)
    ]

def mean_wait(arrivals, services, number_of_agents):
    records = simulate_queue(arrivals, services, number_of_agents)
    waits = [record["wait"] for record in records]
    return statistics.mean(waits) if waits else 0.0


In [4]:
# A small inherited check: more agents cannot delay a caller when
# arrivals and service requirements are held fixed in this FIFO model.
check_arrivals = [0.0, 1.0, 2.5, 2.7, 3.0]
check_services = [2.0, 2.5, 1.0, 3.0, 0.5]
waits_one = [
    row["wait"] for row in simulate_queue(check_arrivals, check_services, 1)
]
waits_two = [
    row["wait"] for row in simulate_queue(check_arrivals, check_services, 2)
]
assert all(two <= one for one, two in zip(waits_one, waits_two))
print("Inherited queue check passed.")


Inherited queue check passed.


## 2. One independent repetition is one complete morning

For morning $r$, let


$$
Y_c^{(r)}=\text{mean waiting time with }c\text{ agents},
\qquad
D_r=Y_{16}^{(r)}-Y_{17}^{(r)}.
$$

A positive $D_r$ means that the seventeenth agent reduced the morning's mean wait. Our target is to measure

$$
\delta=\mathbb E[D_r].
$$

One morning contains roughly 300 caller waits, but they share the same demand conditions and queue. They are dependent outcomes inside one repetition. We use independent Monte Carlo repetitions of complete mornings generated with fresh random inputs.

**Task 1 — complete and trace the paired repetition.** Complete the function in the next cell. It must:

1. generate one morning's arrival times and its daily multiplier;
2. generate exactly one service requirement for every arrival;
3. simulate that same morning with 16 agents and with 17 agents; and
4. assign the two mean waits to `wait_16` and `wait_17`.

The `arrivals` and `services` arrays must be identical in the two queue simulations: only the number of agents changes. Before running the cell, explain why generating a separate morning for each staffing level would give a less precise comparison.


In [5]:
def one_parametric_replication(arrival_rng, service_rng):
    arrivals, multiplier = generate_parametric_morning(arrival_rng)
    services = generate_services(len(arrivals), service_rng)

    # Same arrivals and services for both staffing levels: only agents change.
    wait_16 = mean_wait(arrivals, services, 16)
    wait_17 = mean_wait(arrivals, services, 17)

    return {
        "calls": len(arrivals),
        "multiplier": multiplier,
        "wait_16": wait_16,
        "wait_17": wait_17,
        "difference": wait_16 - wait_17,
    }

arrival_rng = random.Random(20260301)
service_rng = random.Random(20260302)
first_five = [
    one_parametric_replication(arrival_rng, service_rng)
    for _ in range(5)
]

print(f"{'Morning':>7} {'Calls':>6} {'Wait 16':>9} {'Wait 17':>9} {'D_r':>8}")
for morning, result in enumerate(first_five, start=1):
    print(
        f"{morning:>7} {result['calls']:>6} "
        f"{result['wait_16']:>9.2f} {result['wait_17']:>9.2f} "
        f"{result['difference']:>8.2f}"
    )

Morning  Calls   Wait 16   Wait 17      D_r
      1    313      2.23      1.44     0.79
      2    332      1.44      0.66     0.78
      3    275      0.66      0.32     0.34
      4    291      0.75      0.32     0.43
      5    408      6.54      3.94     2.60


The effect size D_r moves with the load. It ranges from 0.34 to 2.60 minutes. On quiet mornings (3 and 4), both staffing levels cope and the gain is small. On the busy morning (5), the extra agent matters most.

**Check your interpretation.** The seed initializes a random stream once. Each call to `one_parametric_replication` advances that stream and creates a new morning. Resetting either seed inside the function would reproduce the same random values instead of generating independent repetitions.

The same `arrivals` and `services` arrays are passed to both staffing levels. That is the paired comparison. Only `number_of_agents` changes.


## 3. Run the outer Monte Carlo loop

We generate 2,000 independent mornings so that we can later inspect how the estimate changes with the number of exerpiments $R$. The lecture's 200-repetition result is the first 200 rows of this reproducible experiment.


In [6]:
def run_parametric_experiment(
    repetitions,
    arrival_seed=20260301,
    service_seed=20260302,
):
    arrival_rng = random.Random(arrival_seed)
    service_rng = random.Random(service_seed)
    return [
        one_parametric_replication(arrival_rng, service_rng)
        for _ in range(repetitions)
    ]

parametric_replications = run_parametric_experiment(2_000)
first_200 = parametric_replications[:200]

assert len(parametric_replications) == 2_000
assert all(
    math.isclose(
        row["difference"],
        row["wait_16"] - row["wait_17"],
        abs_tol=1e-12,
    )
    for row in parametric_replications
)
print("Generated", len(parametric_replications), "independent paired mornings.")


Generated 2000 independent paired mornings.


**Task 2 — identify the two loops.** In your own words, distinguish:

1. the customer loop inside `simulate_queue`, and
2. the morning loop inside `run_parametric_experiment`.

Which loop creates the independent values used to estimate Monte Carlo error? What would go wrong if the random generators were recreated with the same seeds inside that loop?


* The first one, `simulate_queue`, is simulating what actually happens during one morning. Like, each caller's wait time depends on the person before them, so you can't treat them independently.

* The second loop, `run_parametric_experiment`, is where we run that whole morning over and over again. *These* runs are independent from each other, which is exactly what we need for Monte Carlo.

 * If we recreate the seeds inside the loop, every single pass just reproduces the same morning. I tested it and `D_r` came out 0.79 five times in a row — so the SD and standard error were both 0, which makes the results look falsely precise.


## 4. Describe the distribution of morning outcomes

The standard deviation and percentiles below describe how simulated mornings differ. They do not yet describe the uncertainty in the estimated mean.


In [7]:
def linear_quantile(values, probability):
    ordered = sorted(values)
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight

waits_16 = [row["wait_16"] for row in first_200]
waits_17 = [row["wait_17"] for row in first_200]
differences = [row["difference"] for row in first_200]

print(f"{'Measure':<28} {'16 agents':>10} {'17 agents':>10} {'D_r':>10}")
print(
    f"{'Mean (min)':<28} "
    f"{statistics.mean(waits_16):>10.2f} "
    f"{statistics.mean(waits_17):>10.2f} "
    f"{statistics.mean(differences):>10.2f}"
)
print(
    f"{'Standard deviation (min)':<28} "
    f"{statistics.stdev(waits_16):>10.2f} "
    f"{statistics.stdev(waits_17):>10.2f} "
    f"{statistics.stdev(differences):>10.2f}"
)
print(
    f"{'10th percentile of D_r':<28} {'':>10} {'':>10} "
    f"{linear_quantile(differences, 0.10):>10.2f}"
)
print(
    f"{'90th percentile of D_r':<28} {'':>10} {'':>10} "
    f"{linear_quantile(differences, 0.90):>10.2f}"
)


Measure                       16 agents  17 agents        D_r
Mean (min)                         4.35       2.75       1.60
Standard deviation (min)           4.79       3.52       1.38
10th percentile of D_r                                   0.12
90th percentile of D_r                                   3.80


**Task 3 — interpret the spread.** Write two sentences:

1. Interpret the estimated standard deviation of $D_r$ and the 10th–90th percentile range as variation between simulated mornings.

* The SD of 1.38 min means every mornings saving differs from the average saving (1.6) by 1.38 min. About 80% of the mornings give a saving between 0.12 and 3.80 mins.

2. Explain why simulating more mornings estimates this distribution more accurately but does not make individual mornings less variable.

* Simulating more mornings gives better: mean, SD and percentile settle down, however the spread comes from the randomness so SD stays still about 1.4 mins.


## 5. Estimate Monte Carlo error

The sample mean
$$
\bar D_R=\frac{1}{R}\sum_{r=1}^{R}D_r
$$
estimates $\delta=\mathbb E[D_r]$. The standard deviation $s_D$ describes variation between mornings. The standard error
$$
\operatorname{SE}(\bar D_R)=\frac{s_D}{\sqrt R}
$$
describes the Monte Carlo precision of the estimated mean.

**Code to complete.** In the next cell, use `differences` to calculate:

- `mean_difference`, the sample mean of the $D_r$ values;
- `sd_difference`, their sample standard deviation;
- `standard_error`, the estimated standard error of the mean; and
- `confidence_interval`, the two endpoints $\bar D_R \pm 1.96\,\operatorname{SE}(\bar D_R)$.


In [9]:
repetitions = len(differences)

mean_difference = statistics.mean(differences)
sd_difference = statistics.stdev(differences)
standard_error = sd_difference / math.sqrt(repetitions)
confidence_interval = (
    mean_difference - 1.96 * standard_error,
    mean_difference + 1.96 * standard_error,
)


print(f"R: {repetitions}")
print(f"Estimated mean reduction: {mean_difference:.3f} min")
print(f"SD across mornings:       {sd_difference:.3f} min")
print(f"Standard error:           {standard_error:.3f} min")
print(
    "Approximate 95% CI for delta: "
    f"[{confidence_interval[0]:.2f}, {confidence_interval[1]:.2f}] min"
)


R: 200
Estimated mean reduction: 1.604 min
SD across mornings:       1.385 min
Standard error:           0.098 min
Approximate 95% CI for delta: [1.41, 1.80] min


* SD (1.385): how much one morning's saving varies.
* SE (0.098): how precise the average is. It is the SD divided by √200 ≈ 14.1, so it is about 14 times smaller.
* 95% CI [1.41, 1.80]: a plausible range for the true expected saving δ.

**Task 4 — separate the two uncertainties.** Why is the standard error much smaller than the standard deviation? What does it mean that the complete confidence interval is above zero? State the conclusion conditionally on the fixed model.


The SE is way smaller than the SD because we’re averaging over 200 independent mornings, that cancels out a lot of the noise, so the SE shrinks roughly like 1/√R. The SD is just describing a single morning, it doesn’t shrink.

* The simulation shows the 17th agent reduces the expected mean wait, and that is not just Monte Carlo noise.
* Under the fixed Week 2 model, the 17th agent cuts the expected mean wait by around 1.6 minutes, with a 95% CI of 1.41 to 1.80.

## 6. See the law of large numbers and central limit theorem together

The **law of large numbers** explains why $\bar D_R$ settles toward $\delta=\mathbb E[D_r]$ as the number of independent mornings grows.

For sufficiently large $R$, the **central limit theorem** gives the approximation
$$
\frac{\bar D_R-\delta}{s_D/\sqrt R}\approx\mathcal N(0,1),
$$
which leads to the interval $\bar D_R\pm1.96s_D/\sqrt R$.

In [10]:
print(f"{'R':>6} {'Mean D':>10} {'SD D':>10} {'SE':>10} {'95% CI':>23}")
all_differences = [row["difference"] for row in parametric_replications]
for size in (10, 20, 50, 100, 200, 500, 1_000, 2_000):
    current = all_differences[:size]
    current_mean = statistics.mean(current)
    current_sd = statistics.stdev(current)
    current_se = current_sd / math.sqrt(size)
    lower = current_mean - 1.96 * current_se
    upper = current_mean + 1.96 * current_se
    print(
        f"{size:>6} {current_mean:>10.3f} {current_sd:>10.3f} "
        f"{current_se:>10.3f} [{lower:>7.3f}, {upper:>7.3f}]"
    )

     R     Mean D       SD D         SE                  95% CI
    10      1.465      1.341      0.424 [  0.634,   2.296]
    20      1.412      1.219      0.273 [  0.877,   1.946]
    50      1.666      1.359      0.192 [  1.289,   2.043]
   100      1.525      1.341      0.134 [  1.262,   1.788]
   200      1.604      1.385      0.098 [  1.413,   1.796]
   500      1.581      1.383      0.062 [  1.460,   1.703]
  1000      1.478      1.373      0.043 [  1.393,   1.563]
  2000      1.508      1.386      0.031 [  1.447,   1.568]


**Task 5 — read the convergence table.** Describe separately what happens to:

- the running estimate $\bar D_R$;
- the estimated standard deviation $s_D$; and
- the standard error $s_D/\sqrt R$.
Why can a running mean that looks stable after a few repetitions still be misleading?


---

* The running mean fluctuates a lot early on and then starts to settle around 1.5 as R grows. 
* The SD stays around 1.3–1.4 regardless of R, because it's measuring variation between mornings, that doesn't change with more repetitions. 
* The SE on the other hand decreases like 1/√R, going from 0.42 at R = 10 down to 0.03 at R = 2,000. 

A running mean that looks stable after just a few repetitions can still be misleading, because the SE and CI are still large at that point.

## 7. Plan the number of repetitions

If a pilot run estimates the standard deviation as $s_D$, an approximate repetition count for a target confidence-interval half-width $h$ is
$$
R\approx\left(\frac{1.96s_D}{h}\right)^2.
$$
This is a planning approximation. Check the achieved half-width after running the planned experiment because the pilot standard deviation is itself estimated.

In [11]:
def planned_repetitions(pilot_sd, target_half_width):
    return math.ceil((1.96 * pilot_sd / target_half_width) ** 2)

pilot_differences = all_differences[:30]
pilot_sd = statistics.stdev(pilot_differences)

print(f"Pilot SD from 30 mornings: {pilot_sd:.3f} min")
print(f"{'Target half-width':>18} {'Planned R':>12}")
for half_width in (0.50, 0.25, 0.20, 0.10):
    print(
        f"{half_width:>16.2f} min "
        f"{planned_repetitions(pilot_sd, half_width):>12}"
    )


Pilot SD from 30 mornings: 1.333 min
 Target half-width    Planned R
            0.50 min           28
            0.25 min          110
            0.20 min          171
            0.10 min          683


**Task 6 — explain the square.** Why does halving the target half-width require approximately four times as many repetitions? Choose a half-width that would be adequate for a decision you define, and justify that precision before calculating its required $R$.

The half-width shrinks like 1/√R. To make it half as large, √R must double, so R must be 2² = 4 times larger.

* I chose a half-width of h = 0.20 min. The reasoning is that the estimated reduction is around 1.5–1.6 min, which is already about 0.5 min above the threshold. A half-width of 0.20 min is enough to keep the entire CI above 1 minute, meaning Monte Carlo noise won’t flip the decision. Going smaller, like h = 0.10, would require roughly four times as many mornings without actually changing the conclusion.



## 8. Repeat the analysis with the whole-morning bootstrap

One bootstrap repetition selects one observed training morning with replacement, retains its eight ordered quarter-hour counts, places calls uniformly inside each interval, generates new service requirements, and compares 16 with 17 agents using the same arrivals and services.

Selecting a whole morning preserves its overall quiet or busy level and its ordered count profile. There are only ten distinct training profiles, and the exact within-interval arrival times are still simulated rather than observed.


In [12]:
def one_bootstrap_replication(arrival_rng, service_rng):
    arrivals, selected_date = generate_bootstrap_morning(arrival_rng)
    services = generate_services(len(arrivals), service_rng)
    wait_16 = mean_wait(arrivals, services, 16)
    wait_17 = mean_wait(arrivals, services, 17)
    return {
        "date": selected_date,
        "calls": len(arrivals),
        "wait_16": wait_16,
        "wait_17": wait_17,
        "difference": wait_16 - wait_17,
    }

def run_bootstrap_experiment(
    repetitions,
    arrival_seed=20260303,
    service_seed=20260304,
):
    arrival_rng = random.Random(arrival_seed)
    service_rng = random.Random(service_seed)
    return [
        one_bootstrap_replication(arrival_rng, service_rng)
        for _ in range(repetitions)
    ]

bootstrap_replications = run_bootstrap_experiment(200)

print(
    f"{'Arrival generator':<27} {'Wait 16':>9} {'Wait 17':>9} "
    f"{'Mean D':>9} {'95% CI for delta':>22}"
)
for label, results in (
    ("Parametric variable day", first_200),
    ("Whole-morning bootstrap", bootstrap_replications),
):
    wait_16 = [row["wait_16"] for row in results]
    wait_17 = [row["wait_17"] for row in results]
    effect = [row["difference"] for row in results]
    effect_mean = statistics.mean(effect)
    effect_se = statistics.stdev(effect) / math.sqrt(len(effect))
    lower = effect_mean - 1.96 * effect_se
    upper = effect_mean + 1.96 * effect_se
    print(
        f"{label:<27} {statistics.mean(wait_16):>9.2f} "
        f"{statistics.mean(wait_17):>9.2f} {effect_mean:>9.2f} "
        f"[{lower:>7.2f}, {upper:>7.2f}]"
    )


Arrival generator             Wait 16   Wait 17    Mean D       95% CI for delta
Parametric variable day          4.35      2.75      1.60 [   1.41,    1.80]
Whole-morning bootstrap          5.50      3.65      1.86 [   1.68,    2.03]


**Task 7 — separate Monte Carlo error from model sensitivity.** Compare the two confidence intervals and the two estimated effect sizes.

- What conclusion is common to both arrival generators? 
Both CIs are entirely above zero, and both estimates are positive and about 1.6–1.9 min. The 17th agent reduces the expected wait under either arrival model.

- What changes when the generator changes?
The effect size, 1.60 vs 1.86 min. The CIs barely overlap, and waits are higher under the bootstrap (5.50 vs 4.35 at 16 agents).

- Why would increasing both experiments to 20,000 repetitions not resolve the difference between the two generators?
More repetitions only reduce Monte Carlo error, so each generator's estimate becomes more precise. But the two generators represent different demand assumptions, so they converge to different expected effects (about 1.5 vs 1.8 min). That gap is model sensitivity and doesn't shrink with R.

EPLANATION OF 9TH SECTION: 

Section 9 shows that this method works for any morning-level output, not just mean wait. You change the output (here, the fraction of callers who wait), keep the same design, and get a new estimate with its own CI.

The independence is checked by the seed experiment in Task 2: if you reset the seeds, every morning becomes identical and the SE collapses to zero.


## 9. Apply the same design to another output

Choose one morning-level output from Week 2: the 90th percentile of caller waits, the fraction of callers who wait, time-weighted mean queue length, the fraction of the morning with all agents busy, or unfinished calls at 11:00.

**Task 8.** Define precisely what one value for one morning means. Then adapt the paired experiment so that each repetition returns the 16-minus-17-agent difference for that output. Do not pool all callers from all mornings: that would weight busier mornings more heavily and estimate a different quantity.


* I chose the fraction of callers who wait as the output.

* One value for one morning means: the share of that morning's callers whose wait is above 0. It is one number per morning, so the repetition unit is still the morning. 

In [13]:
def fraction_waiting(arrivals, services, number_of_agents):
    records = simulate_queue(arrivals, services, number_of_agents)
    if not records:
        return 0.0
    return sum(record["wait"] > 0 for record in records) / len(records)

def one_fraction_replication(arrival_rng, service_rng):
    arrivals, _ = generate_parametric_morning(arrival_rng)
    services = generate_services(len(arrivals), service_rng)
    frac_16 = fraction_waiting(arrivals, services, 16)
    frac_17 = fraction_waiting(arrivals, services, 17)
    return {"frac_16": frac_16, "frac_17": frac_17, "difference": frac_16 - frac_17}

arrival_rng = random.Random(20260301)
service_rng = random.Random(20260302)
fraction_replications = [
    one_fraction_replication(arrival_rng, service_rng) for _ in range(200)
]
d = [row["difference"] for row in fraction_replications]
mean_d = statistics.mean(d)
se_d = statistics.stdev(d) / math.sqrt(len(d))
print(f"Fraction waiting, 16 agents: {statistics.mean(r['frac_16'] for r in fraction_replications):.3f}")
print(f"Fraction waiting, 17 agents: {statistics.mean(r['frac_17'] for r in fraction_replications):.3f}")
print(f"Mean reduction: {mean_d:.3f}  SE: {se_d:.3f}  95% CI: [{mean_d-1.96*se_d:.3f}, {mean_d+1.96*se_d:.3f}]")
print("All differences >= 0:", all(x >= 0 for x in d))

Fraction waiting, 16 agents: 0.639
Fraction waiting, 17 agents: 0.540
Mean reduction: 0.100  SE: 0.005  95% CI: [0.090, 0.110]
All differences >= 0: True


About 64% of callers wait with 16 agents and 54% with 17. The 17th agent cuts the share of waiting callers by about 10 percentage points, 95% CI 0.09 to 0.11. It is the same design as before: same arrivals and services for both staffing levels, one paired difference per morning, and a standard error from those differences.

## 10. Use an LLM to audit the experiment design

You may give an LLM the replication functions and the following request.

> Audit whether this experiment estimates a paired 16-versus-17-agent effect. Check whether repetitions use fresh random draws, whether both staffing levels receive identical arrivals and services, whether the replication unit is the morning, and whether the standard error uses the paired differences. For each concern, propose a minimal reproducible check and its expected result.

**Task 9.** For every useful proposed check:

1. calculate or reason out the expected result independently;
2. trace the claim to specific code or stored output;
3. run the check and retain it if it protects the experiment; and
4. record unsupported claims and corrections.

A confident explanation is not evidence until its predicted checks have independently verified outcomes.


## Task 9 — Audit of the experiment design

**Method.** I audited the replication functions against the four requested concerns. For each one, I predicted the result, ran a check, and traced the claim to the code.

| # | Concern | Expected result | Observed result | Code traced | Verdict |
|---|---|---|---|---|---|
| 1 | Repetitions use fresh random draws | Values are mostly distinct, and rerunning with the same seeds reproduces the run | 1,997 unique values out of 2,000; the rerun with the same seeds is identical | `run_parametric_experiment` creates both generators once, outside the loop | Supported |
| 2 | 16 and 17 agents receive identical arrivals and services | The two `simulate_queue` calls receive identical inputs and differ only in the agent count | Same arrivals: True; same services: True; agents: 16 and 17 | `one_parametric_replication` generates the arrays once and passes them to both `mean_wait` calls | Supported |
| 3 | The replication unit is the morning | One row per morning, with a varying number of calls | 2,000 rows for 2,000 mornings; calls range from 165 to 522 | Each row of `parametric_replications` holds one `D_r` | Supported |
| 4 | The standard error uses the paired differences | The paired SE is much smaller than an SE that wrongly treats the two waits as independent | Paired SE = 0.031; unpaired SE = 0.142; corr(wait_16, wait_17) = 0.989 | `standard_error = sd_difference / sqrt(R)`, computed from the `D_r` values | Supported |

**Why the paired SE is smaller.** The two waits move together because both are computed on the same morning. Pairing cancels that shared noise, so the SE is about 4.6 times smaller than an unpaired one.

**Checks retained.** All four checks protect the experiment, so I keep them:
- rerun with the same seeds gives the same output;
- identical arrays go to both staffing levels;
- one result row per morning;
- the standard error is computed from paired differences.

**Unsupported claims and corrections.** No unsupported claims were found. One correction applies to wording: "more agents never increase a caller's wait" is verified only when arrivals and services are identical in this FIFO model. It should not be stated for different random inputs.

**Limit of the audit.** These checks show the experiment is implemented as intended. They do not show the model assumptions are realistic.

## 11. State the supported conclusion

**Task 10.** Write a short report with four parts:

1. **Expected effect:** report $\bar D_R$ and its approximate 95% confidence interval for the parametric generator.
2. **Morning variability:** report the standard deviation or 10th–90th percentile range of (D_r), and distinguish it from the standard error.
3. **Sensitivity:** compare the parametric and whole-morning bootstrap results without combining their Monte Carlo intervals.
4. **Boundary:** state why narrow Monte Carlo intervals do not validate the arrival model, service-time assumption, routing, abandonment behavior, or staffing recommendation.

**Next week:** estimate and check the service-time distribution instead of assuming an exponential mean of six minutes.


## Task 10 — Report

**1. Expected effect.** The 17th agent reduces the expected mean wait by **1.60 min** (R = 200, parametric generator), with an approximate 95% CI of **[1.41, 1.80]**. The CI is entirely above zero, so the effect is larger than Monte Carlo noise.

**2. Morning variability.** The SD of D_r is **1.39 min**, and the 10th–90th percentile range is **[0.12, 3.80]**. This is variation between mornings. The standard error (0.098) is much smaller, because it measures the precision of the mean.

**3. Sensitivity.** The whole-morning bootstrap gives **1.86 min**, CI **[1.68, 2.03]**. Both generators agree on the direction of the effect but differ in size. This gap is model sensitivity, not Monte Carlo error, so the intervals are not combined.

**4. Boundary.** A narrow Monte Carlo interval shows precision, not validity. The arrival model, exponential service times, FIFO routing, no abandonment and the staffing decision itself are all untested assumptions.